# ── Cell 7 — MAE gate (advisory, never stops execution) ──────────────────
reg = read_registry()
prod_mae = reg.get('mae') or float('inf')
prev_mae = reg.get('previous_mae')

print('── MAE Gate Assessment ──────────────────────────────────────')
print(f'  New centroid MAE : {mae:.4f} cells')
print(f'  Current prod     : {prod_mae:.4f} cells')
print(f'  Gate threshold   : {prod_mae * 0.98:.4f}')
print()

if prod_mae == float('inf'):
    GATE_STATUS = 'first_model'
    print('✅ First model — will deploy.')
elif mae < prod_mae * 0.98:
    GATE_STATUS = 'passed'
    print(f'✅ GATE PASSED: {(prod_mae-mae)/prod_mae*100:.1f}% improvement. Will deploy.')
elif mae < prod_mae:
    GATE_STATUS = 'marginal'
    print(f'⚠️  GATE MARGINAL: {(prod_mae-mae)/prod_mae*100:.1f}% better but below 2% threshold.')
    print('   Generate more data and retrain.')
else:
    GATE_STATUS = 'failed'
    print(f'❌ GATE FAILED: new MAE {mae:.4f} vs prod {prod_mae:.4f}')
    print()
    print('   Possible causes:')
    print('   • Previous prod MAE was from a leaked model — reset registry:')
    print('     python3 -c "from google.cloud import storage,json as j; ...")  # cloud.sh option 5')
    print('   • Not enough new data — generate more rows')
    print('   • Feature distribution changed — run cloud.sh option 8')
    print()
    print('   Cells 8-9 skipped. Prod model unchanged.')

if WANDB_ACTIVE and run:
    wandb.log({'gate_status': GATE_STATUS, 'prod_mae': prod_mae})

print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 1 — Config + auth + registry ───────────────────────────────────
PROJECT_ID = 'gas-sim-pro-ii'
BUCKET     = f'{PROJECT_ID}-gas-sim-data'

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

gcs    = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

def read_registry():
    return json.loads(bucket.blob('model_registry.json').download_as_text())

reg = read_registry()

print('── Registry ──────────────────────────────')
print(f'  last_trained:     {reg["last_trained"]}')
print(f'  last_data_upload: {reg["last_data_upload"]}')
print(f'  feature_version:  {reg["feature_version"]}')
print(f'  current MAE:      {reg["mae"]}')
print(f'  previous MAE:     {reg["previous_mae"]}')
print(f'  rows_trained_on:  {reg["rows_trained_on"]}')
print(f'  gate_status:      {reg["gate_status"]}')
print('──────────────────────────────────────────')


In [ ]:
# ── Cell 2 — Install dependencies ───────────────────────────────────────
!pip install -q xgboost wandb pyarrow pandas-gbq
print('Done.')


In [ ]:
# ── Cell 3 — Load features from GCS ─────────────────────────────────────
import pandas as pd

blobs = list(gcs.bucket(BUCKET).list_blobs(prefix='features/latest/'))
paths = [f'gs://{BUCKET}/{b.name}' for b in blobs if b.name.endswith('.parquet')]
print(f'Found {len(paths)} parquet file(s)')
assert len(paths) > 0, 'No Parquet files found — run cloud.sh option 8 first'

df = pd.concat(
    [pd.read_parquet(p, storage_options={'token': 'google_default'}) for p in paths],
    ignore_index=True
)

assert len(df) >= 500, f'Only {len(df)} rows — need at least 500'
print(f'Loaded {len(df):,} rows · {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
print()
print('Signal quality (sensor_delta):')
print(f'  min:  {df["sensor_delta"].min():.4f}')
print(f'  mean: {df["sensor_delta"].mean():.4f}')
print(f'  max:  {df["sensor_delta"].max():.4f}')
if 'n_walls' in df.columns:
    print(f'Wall features present: n_walls mean={df["n_walls"].mean():.1f}')


In [ ]:
# ── Cell 4 — Feature matrix + train/val split ────────────────────────────
from sklearn.model_selection import train_test_split
import numpy as np, pandas as pd

TARGETS = ['target_centroid_row', 'target_centroid_col']

BASE_FEATURES = [
    'wind_angle', 'wind_magnitude', 'wind_x', 'wind_y',
    'diffusion_rate', 'decay_factor', 'leak_injection',
    'sensor_count', 'sensor_delta', 'sensor_mean',
    'coverage_ratio', 'reading_variance',
    'centroid_row', 'centroid_col',
    'distance_to_boundary',
]
OPTIONAL = [
    # Spatial shape
    'max_reading_row', 'max_reading_col', 'max_reading',
    'n_sensors_above_threshold',
    # Top-3 sensor positions
    'top1_row', 'top1_col', 'top1_reading',
    'top2_row', 'top2_col', 'top2_reading',
    'top3_row', 'top3_col', 'top3_reading',
    # Triangulation
    'top3_centroid_row', 'top3_centroid_col',
    't1_t2_ratio', 't1_t3_ratio',
    't1_t2_dist', 't1_t3_dist',
    't1_t2_vec_row', 't1_t2_vec_col',
    # Wall/door layout
    'n_walls', 'n_doors',
    'wall_centroid_row', 'wall_centroid_col',
    'wall_spread_row', 'wall_spread_col',
    'walls_q1', 'walls_q2', 'walls_q3', 'walls_q4',
    'wall_density', 'open_path_ratio',
    'walls_near_centroid', 'walls_blocking_top1',
]
FEATURES = BASE_FEATURES + [f for f in OPTIONAL if f in df.columns]
print(f'Using {len(FEATURES)} features ({len(BASE_FEATURES)} base + {len(FEATURES)-len(BASE_FEATURES)} optional)')

df_feat = df[FEATURES].copy()

# ── Derived features ──────────────────────────────────────────────────────
df_feat['wind_corr_row'] = df['centroid_row'] - df['wind_y'] * 5
df_feat['wind_corr_col'] = df['centroid_col'] - df['wind_x'] * 5
if 'max_reading_row' in df.columns:
    df_feat['disp_row'] = df['max_reading_row'] - df['centroid_row']
    df_feat['disp_col'] = df['max_reading_col'] - df['centroid_col']

# Triangulation — compute if not already in Parquet
if 'top1_reading' in df.columns and 'top3_centroid_row' not in df.columns:
    total = df['top1_reading'] + df['top2_reading'] + df['top3_reading'] + 1e-9
    df_feat['top3_centroid_row'] = (
        df['top1_row']*df['top1_reading'] +
        df['top2_row']*df['top2_reading'] +
        df['top3_row']*df['top3_reading']) / total
    df_feat['top3_centroid_col'] = (
        df['top1_col']*df['top1_reading'] +
        df['top2_col']*df['top2_reading'] +
        df['top3_col']*df['top3_reading']) / total
if 'top1_reading' in df.columns and 't1_t2_ratio' not in df.columns:
    df_feat['t1_t2_ratio'] = df['top1_reading'] / (df['top2_reading'] + 1e-9)
    df_feat['t1_t3_ratio'] = df['top1_reading'] / (df['top3_reading'] + 1e-9)
if 'top1_row' in df.columns and 't1_t2_dist' not in df.columns:
    df_feat['t1_t2_dist'] = np.sqrt(
        (df['top1_row']-df['top2_row'])**2 + (df['top1_col']-df['top2_col'])**2)
    df_feat['t1_t3_dist'] = np.sqrt(
        (df['top1_row']-df['top3_row'])**2 + (df['top1_col']-df['top3_col'])**2)
    df_feat['t1_t2_vec_row'] = df['top1_row'] - df['top2_row']
    df_feat['t1_t2_vec_col'] = df['top1_col'] - df['top2_col']

# Wall-derived features
if 'n_walls' in df.columns:
    df_feat['wall_asymmetry_row'] = df['walls_q1'] + df['walls_q2'] - df['walls_q3'] - df['walls_q4']
    df_feat['wall_asymmetry_col'] = df['walls_q1'] + df['walls_q3'] - df['walls_q2'] - df['walls_q4']
    df_feat['leak_to_wall_centroid_row'] = df['centroid_row'] - df['wall_centroid_row']
    df_feat['leak_to_wall_centroid_col'] = df['centroid_col'] - df['wall_centroid_col']

# ── Sanitise — clip and replace inf/nan ──────────────────────────────────
# Clip ratio features — near-zero denominators produce huge values
for col in ['t1_t2_ratio', 't1_t3_ratio']:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 100)

# Clip distance features to grid diagonal (~141 cells)
for col in ['t1_t2_dist', 't1_t3_dist']:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].clip(0, 142)

# Final safe X build
X = df_feat.replace([np.inf, -np.inf], 0).fillna(0).values.astype(np.float32)
y = df[TARGETS].values.astype(np.float32)

# Verify no inf/nan
assert not np.isinf(X).any(), 'inf in X after sanitisation'
assert not np.isnan(X).any(), 'nan in X after sanitisation'

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')
print(f'Total features: {X.shape[1]}')


In [ ]:
# ── Cell 5 — W&B init (optional, never blocks) ───────────────────────────
import wandb

WANDB_ACTIVE = False
run = None
try:
    run = wandb.init(
        project='gas-sim-pro',
        anonymous='allow',
        mode='offline',
        config={
            'model': 'xgboost',
            'features': FEATURES,
            'n_rows': len(df),
            'feature_version': reg['feature_version'],
        }
    )
    WANDB_ACTIVE = True
    print('W&B run initialised (offline mode)')
except Exception as e:
    print(f'W&B unavailable ({e}) — continuing without tracking')


In [ ]:
# ── Cell 6 — Train XGBoost ───────────────────────────────────────────────
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

BASE_PARAMS = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)

model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**BASE_PARAMS))
model_centroid.fit(X_tr, y_tr)

y_pred = model_centroid.predict(X_val)
mae    = float(np.mean(np.abs(y_pred - y_val)))

# Aliases for downstream cells
model  = model_centroid
mae_centroid = mae
mae_nearest  = mae
mae_count    = 0.0

if WANDB_ACTIVE and run:
    wandb.log({'val_mae': mae})

print(f'Val MAE: {mae:.4f} cells')
print(f'Primary metric (centroid MAE): {mae:.4f}')


In [ ]:
# ── Cell 6b — Optuna hyperparameter search (optional) ────────────────────
# Set RUN_OPTUNA=True to search. Runs ~20-40 min on Colab free tier.
# Best params saved to GCS and reused automatically on future runs.

RUN_OPTUNA = False
N_TRIALS   = 40

if RUN_OPTUNA:
    !pip install -q optuna
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 100, 800),
            max_depth        = trial.suggest_int('max_depth', 3, 10),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            subsample        = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
            gamma            = trial.suggest_float('gamma', 0, 5),
            random_state=42, n_jobs=-1, verbosity=0
        )
        m = MultiOutputRegressor(xgb.XGBRegressor(**params))
        m.fit(X_tr, yc_tr)
        pred = m.predict(X_val)
        return float(np.mean(np.abs(pred - yc_val)))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best = study.best_params
    print(f'Best centroid MAE: {study.best_value:.4f}')
    print(f'Best params: {best}')

    bucket.blob('registry/hparam_best.json').upload_from_string(
        json.dumps({'best_params': best, 'best_mae': study.best_value,
                    'n_trials': N_TRIALS, 'n_rows': len(df),
                    'searched_at': __import__('datetime').datetime.utcnow().isoformat()},
                   indent=2),
        content_type='application/json'
    )
    print('Best params saved to registry/hparam_best.json')

    # Retrain all three models with best params
    model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
    model_centroid.fit(X_tr, yc_tr)
    model_nearest = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
    model_nearest.fit(X_tr, yn_tr)
    model_count = xgb.XGBRegressor(**best, random_state=42, n_jobs=-1)
    model_count.fit(X_tr, yk_tr.ravel())
    pred_c = model_centroid.predict(X_val)
    mae = float(np.mean(np.abs(pred_c - yc_val)))
    print(f'Final centroid MAE with best params: {mae:.4f}')

else:
    try:
        saved = json.loads(bucket.blob('registry/hparam_best.json').download_as_text())
        if saved.get('n_rows', 0) >= len(df) * 0.5:
            best = saved['best_params']
            print(f'Using saved hyperparameters (searched on {saved["n_rows"]:,} rows)')
            model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
            model_centroid.fit(X_tr, yc_tr)
            model_nearest = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
            model_nearest.fit(X_tr, yn_tr)
            model_count = xgb.XGBRegressor(**best, random_state=42, n_jobs=-1)
            model_count.fit(X_tr, yk_tr.ravel())
            pred_c = model_centroid.predict(X_val)
            mae = float(np.mean(np.abs(pred_c - yc_val)))
            print(f'Centroid MAE with saved params: {mae:.4f}')
        else:
            print('Saved params from too few rows — using Cell 6 defaults.')
    except Exception:
        print('No saved hyperparameters — using Cell 6 defaults.')


In [ ]:
# ── Cell 7 — MAE gate (advisory, never stops execution) ──────────────────
reg = read_registry()
prod_mae = reg.get('mae') or float('inf')
prev_mae = reg.get('previous_mae')

print('── MAE Gate Assessment ──────────────────────────────────────')
print(f'  New centroid MAE : {mae:.4f} cells')
print(f'  Current prod     : {prod_mae:.4f} cells')
print(f'  Gate threshold   : {prod_mae * 0.98:.4f}')
print()

if prod_mae == float('inf'):
    GATE_STATUS = 'first_model'
    print('✅ First model — will deploy.')
elif mae < prod_mae * 0.98:
    GATE_STATUS = 'passed'
    print(f'✅ GATE PASSED: {(prod_mae-mae)/prod_mae*100:.1f}% improvement. Will deploy.')
elif mae < prod_mae:
    GATE_STATUS = 'marginal'
    print(f'⚠️  GATE MARGINAL: {(prod_mae-mae)/prod_mae*100:.1f}% better but below 2% threshold.')
    print('   Generate more data and retrain.')
else:
    GATE_STATUS = 'failed'
    print(f'❌ GATE FAILED: {(mae-prod_mae)/prod_mae*100:.1f}% worse than production.')
    print('   Possible causes: stale Parquet, same data, distribution shift.')
    print('   Run cloud.sh option 8 then retry.')
    print('   Cells 8-9 will be skipped. Prod model unchanged.')

if WANDB_ACTIVE and run:
    wandb.log({'gate_status': GATE_STATUS, 'prod_mae': prod_mae})

print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 8 — Export model ────────────────────────────────────────────────
import os, datetime, joblib

if GATE_STATUS not in ('passed', 'first_model'):
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
    VERSION = None
else:
    RUN_ID  = run.id if WANDB_ACTIVE and run else datetime.datetime.now().strftime('%H%M%S')
    VERSION = f"v{datetime.date.today().strftime('%Y%m%d')}-{RUN_ID}"
    os.makedirs(f'/tmp/{VERSION}', exist_ok=True)
    joblib.dump(model_centroid, f'/tmp/{VERSION}/model.joblib')
    print(f'✓ Model exported: {VERSION}')
    print(f'  MAE: {mae:.4f}  Features: {X_tr.shape[1]}')


In [ ]:
# ── Cell 9 — Push to GCS ─────────────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
else:
    blob = bucket.blob(f'models/{VERSION}/model.joblib')
    blob.upload_from_filename(f'/tmp/{VERSION}/model.joblib')
    print(f'✓ Uploaded: models/{VERSION}/model.joblib')
    if WANDB_ACTIVE and run:
        try:
            wandb.log_artifact(f'/tmp/{VERSION}/model.joblib', name='model', type='model')
        except Exception as e:
            print(f'  W&B artifact skipped: {e}')


In [ ]:
# ── Cell 10 — Update registry ────────────────────────────────────────────
import datetime as dt

prev_version = reg.get('latest_version')
prev_mae_val = reg.get('mae')

if GATE_STATUS in ('passed', 'first_model') and VERSION is not None:
    reg.update({
        'latest_version':   VERSION,
        'previous_version': prev_version,
        'last_trained':     dt.datetime.now(dt.timezone.utc).isoformat(),
        'model_path':       f'models/{VERSION}/model_centroid.joblib',
        'joblib_path':      f'models/{VERSION}/model_centroid.joblib',
        'mae':              mae,
        'previous_mae':     prev_mae_val,
        'rows_trained_on':  len(df),
        'feature_version':  reg['feature_version'],
        'gate_status':      GATE_STATUS,
        'mae_nearest':      mae_nearest,
        'mae_count':        mae_count,
        'model_nearest_path': f'models/{VERSION}/model_nearest.joblib',
        'model_count_path':   f'models/{VERSION}/model_count.joblib',
        'n_features':       len(FEATURES),
    })
else:
    reg.update({
        'last_trained':    dt.datetime.now(dt.timezone.utc).isoformat(),
        'gate_status':     GATE_STATUS,
        'rows_trained_on': len(df),
    })

reg_blob = bucket.blob('model_registry.json')
reg_blob.upload_from_string(json.dumps(reg, indent=2), content_type='application/json')
reg_blob.cache_control = 'no-cache, no-store, max-age=0'
reg_blob.patch()

result = {
    'status':      GATE_STATUS,
    'mae':         mae,
    'mae_nearest': mae_nearest,
    'mae_count':   mae_count,
    'prod_mae':    prod_mae,
    'version':     VERSION,
    'trained_at':  dt.datetime.now(dt.timezone.utc).isoformat(),
    'rows':        len(df),
}
r_blob = bucket.blob('registry/last_training_result.json')
r_blob.upload_from_string(json.dumps(result, indent=2, default=str), content_type='application/json')
r_blob.cache_control = 'no-cache, no-store, max-age=0'
r_blob.patch()

if WANDB_ACTIVE and run:
    wandb.finish()

print()
print('── Training Summary ─────────────────────────────────────────')
print(f'  Gate:          {GATE_STATUS}')
print(f'  Centroid MAE:  {mae:.4f}')
print(f'  Nearest MAE:   {mae_nearest:.4f}')
print(f'  Count MAE:     {mae_count:.4f} leaks')
print(f'  Version:       {VERSION}')
print(f'  Rows trained:  {len(df):,}')
print()
if GATE_STATUS in ('passed', 'first_model'):
    print('🚀 Deploy function will trigger automatically (~5 min).')
    print('   TRAIN button will flash green when deployed.')
elif GATE_STATUS == 'marginal':
    print('⚠️  Below threshold. Generate more data and retrain.')
    print('   TRAIN button will flash yellow.')
else:
    print('❌ Prod model unchanged.')
    print('   Run cloud.sh option 8 then retry.')
    print('   TRAIN button will flash yellow.')
print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 11 — Verify predictions ─────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
else:
    sample = X_val[:5]
    pred_c = model_centroid.predict(sample)
    print('Predictions (first 5 rows):')
    print(f'{"centroid row":>14}  {"centroid col":>14}  {"true row":>10}  {"true col":>10}  {"err":>6}')
    for i in range(5):
        err = abs(pred_c[i][0]-y_val[i][0]) + abs(pred_c[i][1]-y_val[i][1])
        print(f'  pred: ({pred_c[i][0]:5.1f}, {pred_c[i][1]:5.1f})'
              f'  true: ({y_val[i][0]:5.1f}, {y_val[i][1]:5.1f})'
              f'  err: {err:.2f}')
    print('Verification complete.')
